# 衛生福利部臺北醫院
## SmartCoder API 介接與逐項測試

**院別：`tph`　｜　API 根網址：`https://smartcodern.itri-nlp.tw/tph`**

本手冊供院方資訊室與 HIS 介接工程師使用。每項 API 都有參數、可執行範例、預期狀態碼及回應說明；按該區塊左側 **▶** 即可測試。

### 開始使用

1. 選擇 Python 3／CPU 執行階段，不需要 GPU。先執行「0. 初始化」與「金鑰設定」。
2. 依目錄逐項測試。編碼 POST 會產生 `request_id`；GET 用同一 ID 回查。也可選「執行階段 → 全部執行」。
3. 金鑰由計畫窗口透過私有管道提供，輸入時隱藏。不要把金鑰寫進程式、網址或儲存的輸出。
4. 預設案例是合成文字：`患者胸痛持續兩週，否認咳嗽。`。可調整參數測試其他合成案例；本公開 Colab 請勿輸入真實病歷或個資。

### API 功能索引

| 區塊 | 方法與路徑 | 用途 | 成功／預期 HTTP | 金鑰 |
|---|---|---|---|---|
| 1 | `GET /healthz` | 確認服務可連線 | 200 | 不需要 |
| 2 | `GET /openapi.json` | 查詢本院介面 schema | 404（此院未公開此路徑） | 需要 |
| 3 | `POST /api/v1/snomed/coding` | 原始文字 → 整理 → SNOMED CT 編碼 | 200 | 需要 |
| 4 | `GET /api/v1/snomed/results/{request_id}` | 以同一 ID 取得完整結果 | 200 | 需要 |
| 5 | 同一 POST，`output_format="fhir"` | 查看 FHIR 格式內容 | 200 | 需要 |
| 錯誤測試 | POST／GET | 無金鑰、錯誤金鑰、查無 ID、參數錯誤 | 401／404／422 | 依案例 |
| CORS | POST／GET 的 `OPTIONS` | 檢查 Colab Origin 預檢 | 200 或 204 | 不需要 |

以下使用本院唯一根網址。`X-API-Key` 放在 header；不需要呼叫端另加 `X-Hospital-Slug`。管理頁面、稽核資料列表及服務內部回呼不屬於院方編碼介接功能，因此本手冊不呼叫這些路徑。

## 0. 初始化

執行一次，載入連線與結果檢查工具。此區塊不送 API 請求；每項請求都寫在後續對應區塊中。重新執行初始化會清除本次記憶體內的測試紀錄。

In [ ]:
#@title 初始化與測試工具（執行一次）
HOSPITAL_SLUG = 'tph'
BASE_URL = 'https://smartcodern.itri-nlp.tw/tph'
REQUEST_FIELDS = ['request_id', 'encounter_type', 'language', 'SOAP', 'tui', 'raw_clinical_note', 'icd_codes', 'output_format', 'department', 'threshold']
OPENAPI_AVAILABLE = False
STREAM_SUPPORTED = False
"""Secret-free client embedded in each hospital notebook; no fallback routes."""
import json
import math
import time
from datetime import datetime
from getpass import getpass
from contextlib import contextmanager
from html import escape

try:
    from IPython.display import display, JSON, HTML
except ImportError:
    # The same request cells can also run in a plain Python verification process.
    def JSON(value, expanded=True):
        return json.dumps(value, ensure_ascii=False, indent=2)
    def HTML(value):
        return value
    def display(value):
        print(value)
from uuid import UUID, uuid4

import requests

COLAB_ORIGIN = "https://colab.research.google.com"
RAW_NOTE = "患者胸痛持續兩週，否認咳嗽。"
KNOWN_SNOMED_FIXTURE = {"29857009": "Chest pain"}
RESULT_FIELDS = {"request_id", "polished_clinical_note", "snomed_codings", "processing_metadata"}
CODING_FIELDS = {"concept_id", "term", "confidence", "source"}
METADATA_FIELDS = {"polish_model", "snomed_version", "pipeline_version", "vote_attempts", "confidence_method", "timestamp"}
GET_FIELDS = {"request_id", "status", "status_code", "occurred_at", "duration_ms", "response", "error_detail"}


def require(condition, message):
    if not condition:
        raise ValueError(message)


def timestamp(value, field):
    require(isinstance(value, str) and bool(value.strip()), f"{field} 必須是時間字串")
    parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
    require(parsed.tzinfo is not None and parsed.utcoffset() is not None, f"{field} 缺少時區")


def cors(response):
    actual = response.headers["Access-Control-Allow-Origin"]
    require(actual in (COLAB_ORIGIN, "*"), "回應未允許 Colab Origin")


def http(response, expected_status, expected_url, label):
    # Do not print error bodies: they may contain internal service details.
    acceptable = (expected_status,) if isinstance(expected_status, int) else expected_status
    require(response.status_code in acceptable,
            f"{label} HTTP {response.status_code}；預期 {expected_status}，已停止")
    require(response.url == expected_url and not response.history, f"{label} 不接受網址轉向")


def result_contract(payload, request_id, body=None):
    known_body = body is not None
    fixture = known_body and body["raw_clinical_note"] == RAW_NOTE and not body.get("tui") and not body.get("icd_codes")
    allowed_fields = RESULT_FIELDS | {"allowed_tui", "fhir_bundle"}
    require(isinstance(payload, dict) and RESULT_FIELDS <= set(payload) <= allowed_fields, "POST 結果欄位不符合契約")
    require(isinstance(payload["request_id"], str) and payload["request_id"] == request_id, "request_id 不一致")
    polished = payload["polished_clinical_note"]
    require(isinstance(polished, str) and bool(polished.strip()), "整理後病歷為空或型別不符")
    if fixture:
        require(" ".join(polished.split()).casefold() != " ".join(body["raw_clinical_note"].split()).casefold(), "未觀察到病歷整理")
    codings = payload["snomed_codings"]
    require(isinstance(codings, list) and (bool(codings) or not fixture), "編碼結果為空或型別不符")
    seen = set()
    for item in codings:
        require(isinstance(item, dict), "編碼項目必須是 object")
        required_coding = CODING_FIELDS - {"confidence"} if HOSPITAL_SLUG == "yuanrung" else CODING_FIELDS
        require(required_coding <= set(item) <= CODING_FIELDS | {"category", "tui", "code_name"}, "編碼項目欄位不符合現行 OpenAPI")
        concept_id = item["concept_id"]
        require(isinstance(concept_id, str) and concept_id.isdigit(), "concept_id 必須是數字字串")
        require(concept_id not in seen, "concept_id 重複")
        # This single synthetic fixture has one positive clinical finding.
        # Membership is checked against its known SNOMED CT concept, not guessed from digits.
        require(not fixture or concept_id in KNOWN_SNOMED_FIXTURE, "回傳概念不在此合成測試的已知 SNOMED CT 集合")
        seen.add(concept_id)
        require(isinstance(item["term"], str) and bool(item["term"].strip()), "term 為空或型別不符")
        confidence = item["confidence"] if "confidence" in item else None
        if confidence is None:
            require((HOSPITAL_SLUG == "yuanrung" and (not known_body or bool(body.get("icd_codes")))) or "ICD10_CROSSWALK" in item["source"], "只有 ICD 對照項目可缺少投票支持度")
        else:
            require(type(confidence) in (int, float) and math.isfinite(confidence) and 0 < confidence <= 1, "confidence 型別或範圍錯誤")
        require(isinstance(item["source"], list) and bool(item["source"]), "source 必須是非空 array")
        require(all(isinstance(v, str) and bool(v) for v in item["source"]), "source 項目型別不符")
        if known_body and not body.get("icd_codes"):
            require(item["source"] == ["TXT_NER"], "編碼來源不符合 TXT_NER 契約")
        if "category" in item:
            require(item["category"] in ("finding", "disorder", "procedure"), "category 值不符")
        if "tui" in item:
            require(isinstance(item["tui"], str) and bool(item["tui"].strip()), "tui 型別或內容不符")
        if "code_name" in item:
            require(isinstance(item["code_name"], str) and bool(item["code_name"].strip()), "code_name 型別或內容不符")
    require(not fixture or seen == set(KNOWN_SNOMED_FIXTURE), "未辨識胸痛；否認咳嗽不得產生陽性編碼")
    metadata = payload["processing_metadata"]
    require(isinstance(metadata, dict) and set(metadata) == METADATA_FIELDS, "處理 metadata 欄位不符")
    require(isinstance(metadata["polish_model"], str) and bool(metadata["polish_model"].strip()), "病歷整理 metadata 缺少模型標記")
    require(metadata["snomed_version"] == "2026-07-01", "SNOMED CT 版本不符")
    require(metadata["pipeline_version"] == "api-version-0.1.0|txt_ner", "完整術語流程版本不符")
    require(type(metadata["vote_attempts"]) is int and metadata["vote_attempts"] == 3, "NER 投票輪數必須是 3")
    require(metadata["confidence_method"] == "txt_ner_assertion_filtered_vote_support_min_1", "投票信心度方法不符")
    timestamp(metadata["timestamp"], "processing_metadata.timestamp")
    if known_body and body.get("tui"):
        require(payload["allowed_tui"] == body["tui"], "allowed_tui 與請求不同")
    if (known_body and body["output_format"] == "fhir") or (not known_body and "fhir_bundle" in payload):
        fhir_contract(payload)
    else:
        require("fhir_bundle" not in payload, "simple 不應包含 fhir_bundle")
    return codings


def get_contract(payload, request_id, post_payload=None, body=None):
    require(isinstance(payload, dict) and set(payload) == GET_FIELDS, "GET 結果欄位不符")
    require(isinstance(payload["request_id"], str) and payload["request_id"] == request_id, "GET request_id 不符")
    require(payload["status"] == "completed", "任務尚未完成")
    require(type(payload["status_code"]) is int and payload["status_code"] == 200, "儲存的任務狀態碼不符")
    require(type(payload["duration_ms"]) is int and payload["duration_ms"] >= 0, "duration_ms 不符")
    timestamp(payload["occurred_at"], "occurred_at")
    require(payload["error_detail"] is None, "GET 結果包含錯誤")
    result_contract(payload["response"], request_id, body)
    if post_payload is not None:
        require(payload["response"] == post_payload, "GET 與 POST 完整結果不一致")


def fhir_contract(payload):
    bundle = payload["fhir_bundle"]
    require(isinstance(bundle, dict) and set(bundle) == {"entry"}, "fhir_bundle 結構不符")
    entries = bundle["entry"]
    require(isinstance(entries, list) and len(entries) == len(payload["snomed_codings"]) + 1, "FHIR entry 數量不符")
    composition = entries[0]["resource"]
    require(composition["resourceType"] == "Composition", "FHIR 首筆必須是 Composition")
    sections = composition["section"]
    require(isinstance(sections, list) and len(sections) == 1, "Composition section 不符")
    require(sections[0]["text"]["status"] == "generated", "Composition text.status 不符")
    require(isinstance(sections[0]["text"]["div"], str) and payload["polished_clinical_note"] in sections[0]["text"]["div"], "Composition 病歷內容不符")
    for entry, coding in zip(entries[1:], payload["snomed_codings"]):
        resource = entry["resource"]
        require(resource["resourceType"] == "Observation", "編碼 resource 必須是 Observation")
        fhir_coding = resource["code"]["coding"]
        require(isinstance(fhir_coding, list) and len(fhir_coding) == 1, "Observation coding 不符")
        require(fhir_coding[0] == {"system": "http://snomed.info/sct", "code": coding["concept_id"], "display": coding["term"]}, "FHIR 與 SNOMED 編碼不一致")


# Helpers only prepare requests, display results, and check response contracts.
# Every API operation is called explicitly in its own notebook cell below.
SESSION = requests.Session()
TESTS = {}
COMPLETED = {}


def show_table(rows):
    columns = ["項目", "結果", "HTTP", "秒數"]
    head = "".join("<th>" + escape(v) + "</th>" for v in columns)
    body = "".join("<tr>" + "".join("<td>" + escape(str(row[k])) + "</td>" for k in columns) + "</tr>" for row in rows)
    display(HTML("<table><thead><tr>" + head + "</tr></thead><tbody>" + body + "</tbody></table>"))


@contextmanager
def test_case(name):
    TESTS[name] = {"項目": name, "結果": "執行中", "HTTP": "", "秒數": ""}
    started = time.perf_counter()
    try:
        yield
    except Exception as exc:
        TESTS[name]["結果"] = "失敗"
        # Original server bodies and third-party exceptions are not printed.
        print(f"{name}：失敗；請核對 HTTP 狀態、參數與本院金鑰後重跑此項。")
        reason = str(exc) if type(exc) is ValueError else type(exc).__name__
        raise RuntimeError(f"{name} 未通過：{reason}") from None
    else:
        TESTS[name]["結果"] = "通過"
        print(name + "：通過")
    finally:
        TESTS[name]["秒數"] = round(time.perf_counter() - started, 2)
        print("本項耗時：", TESTS[name]["秒數"], "秒")


def auth_headers():
    require("API_KEY" in globals() and isinstance(API_KEY, str) and bool(API_KEY), "請先執行金鑰設定")
    return {"X-API-Key": API_KEY, "Content-Type": "application/json; charset=utf-8", "Origin": COLAB_ORIGIN}


def request(method, path, *, body=None, authenticated=True, headers=None, stream=False):
    require(path.startswith("/") and "://" not in path and ".." not in path, "請使用本院相對路徑")
    url = BASE_URL + path
    actual_headers = auth_headers() if authenticated else {"Origin": COLAB_ORIGIN}
    if headers is not None:
        actual_headers = headers
    print(f"{method} {url}")
    print("X-API-Key：已設定（隱藏）" if "X-API-Key" in actual_headers else "X-API-Key：未提供")
    if body is not None:
        display(JSON(body, expanded=True))
    return SESSION.request(method, url, headers=actual_headers, json=body,
                           timeout=(20, 240), allow_redirects=False, stream=stream)


def checked(response, expected, name, *, check_cors=True):
    print("HTTP", response.status_code)
    TESTS[name]["HTTP"] = response.status_code
    http(response, expected, response.request.url, name)
    if check_cors:
        cors(response)


def public_view(value):
    # The API field remains in memory for validation; the displayed identifier is masked.
    if isinstance(value, dict):
        return {k: ("external AI capacity" if k == "polish_model" else
                    [s if s in ("TXT_NER", "ICD10_CROSSWALK") else "terminology mapping" for s in v] if k == "source" and isinstance(v, list) else
                    public_view(v)) for k, v in value.items()}
    if isinstance(value, list):
        return [public_view(v) for v in value]
    return value


def show_json(response):
    display(JSON(public_view(response.json()), expanded=True))


def show_error(response):
    payload = response.json()
    require(isinstance(payload, dict) and "detail" in payload, "錯誤回應缺少 detail")
    if isinstance(payload["detail"], list):
        # Do not echo rejected input values.
        display(JSON({"detail": [{"loc": e["loc"], "type": e["type"]} for e in payload["detail"]]}, expanded=True))
    else:
        require(isinstance(payload["detail"], str) and bool(payload["detail"]), "detail 型別不符")
        print("detail：服務已拒絕此請求")


def preflight(path, method, name):
    response = request("OPTIONS", path, authenticated=False, headers={
        "Origin": COLAB_ORIGIN, "Access-Control-Request-Method": method,
        "Access-Control-Request-Headers": "content-type,x-api-key"})
    checked(response, (200, 204), name)
    methods = {v.strip().upper() for v in response.headers["Access-Control-Allow-Methods"].split(",")}
    allowed = {v.strip().lower() for v in response.headers["Access-Control-Allow-Headers"].split(",")}
    require(method in methods or "*" in methods, "CORS 未允許此方法")
    require({"content-type", "x-api-key"} <= allowed or "*" in allowed, "CORS 未允許必要 headers")
    display(JSON({k: response.headers[k] for k in ["Access-Control-Allow-Origin", "Access-Control-Allow-Methods", "Access-Control-Allow-Headers"]}, expanded=True))


def remember(body, payload):
    COMPLETED[body["request_id"]] = {"body": body, "response": payload}


def completed_coding_response(response, request_id, body, name):
    expected = (200, 202) if HOSPITAL_SLUG == "yuanrung" and body.get("icd_codes") else 200
    checked(response, expected, name)
    if response.status_code == 200:
        return response.json()
    partial = response.json()
    require(set(partial) == {"request_id", "status", "snomed_codings", "result_is_final"}, "202 初步結果欄位不符")
    require(partial["request_id"] == request_id and partial["status"] == "processing" and partial["result_is_final"] is False, "202 尚未完成結果不符")
    require(isinstance(partial["snomed_codings"], list), "初步對照必須是 array")
    print("HTTP 202：已接受；正在等待文字分析完成，初步對照不是最終結果。")
    deadline = time.monotonic() + 240
    while time.monotonic() < deadline:
        saved = request("GET", "/api/v1/snomed/results/" + request_id)
        http(saved, 200, saved.request.url, name)
        cors(saved)
        payload = saved.json()
        require(isinstance(payload, dict) and set(payload) == GET_FIELDS, "GET 任務欄位不符")
        require(payload["request_id"] == request_id, "GET 任務 ID 不符")
        if payload["status"] == "completed":
            get_contract(payload, request_id, body=body)
            TESTS[name]["HTTP"] = "202 → 200"
            return payload["response"]
        require(payload["status"] == "running" and payload["status_code"] == 202, "本案分析失敗或狀態不符，請保留 request_id 聯絡維運")
        time.sleep(1)
    raise ValueError("本案仍未完成，請以相同 request_id 回查；不要重複送出")


def lookup_completed(request_id, name):
    response = request("GET", "/api/v1/snomed/results/" + request_id)
    checked(response, 200, name)
    payload = response.json()
    if request_id in COMPLETED:
        original = COMPLETED[request_id]
        get_contract(payload, request_id, original["response"], original["body"])
        print("GET 與本次 POST 完整 JSON 一致。")
    else:
        get_contract(payload, request_id)
        print("此 ID 未由本次 notebook 送出；已檢查回應欄位，未比對先前 POST。")
    show_json(response)
    return payload

print('已準備本院 API：', BASE_URL)

### 金鑰設定

執行後貼上本院 API key。金鑰只保存在目前執行階段記憶體中；結束時可執行最後的清除區塊。

In [ ]:
API_KEY = getpass(f'請輸入 {HOSPITAL_SLUG} 的 API key（隱藏輸入）：').strip()
require(bool(API_KEY), '尚未輸入金鑰')
print('本院金鑰已設定；內容不顯示。')

## 1. GET／健康檢查

**路徑：`/healthz`　｜　參數：無　｜　金鑰：不需要　｜　預期：HTTP 200**

`status="ok"` 表示 API 程序能回應；此檢查不會執行編碼。`service` 與 `version` 是服務識別及版本。

In [ ]:
name = "健康檢查"
with test_case(name):
    health = request("GET", "/healthz", authenticated=False)
    checked(health, 200, name, check_cors=False)
    health_body = health.json()
    require(isinstance(health_body, dict) and health_body["status"] == "ok", "健康狀態不符")
    show_json(health)

## 2. GET／查詢介面定義

**路徑：`/openapi.json`　｜　參數：無　｜　金鑰：需要**

此院正式入口未開放 OpenAPI 路徑，預期回傳 404；這項測試確認目前公開範圍。請依本手冊的欄位表介接，404 不代表編碼功能故障。

In [ ]:
name = "OpenAPI 公開範圍"
with test_case(name):
    schema_response = request("GET", "/openapi.json")
    checked(schema_response, 404, name, check_cors=False)
    show_error(schema_response)

## 3. POST／病歷文字編碼

**路徑：`/api/v1/snomed/coding`　｜　金鑰：需要　｜　預期：HTTP 200、JSON**

### 請求參數

| 欄位 | 型別 | 本手冊預設／用途 |
|---|---|---|
| `request_id` | string | 每次 POST 產生新的 UUID，供 GET 回查 |
| `raw_clinical_note` | string | 原始合成病歷文字，不可空白 |
| `output_format` | string | `simple` 回傳編碼 JSON；`fhir` 額外回傳 `fhir_bundle` |
| `language` | string，可省略 | 輸入語言的紀錄資訊，例如 `zh-TW` |
| `encounter_type` | string，可省略 | 就醫類型紀錄資訊 |
| `department` | string，可省略或 null | 科別保留欄位，目前不影響編碼 |
| `threshold` | number，可省略或 null | 數值保留欄位，目前不影響信心值或篩選 |
| `SOAP` | string，可省略 | 區段標記，例如 `S`；不是病歷本文 |
| `tui` | array[string]，可省略 | 限定語意類型，例如 `["T184"]`；表單用逗號分隔 |
| `icd_codes` | array[object]，可省略 | 額外 ICD 證據；每項至少有 `code`，可加 `system`、`description` |

空白的選填欄位不會送出。預設只送 `request_id`、`raw_clinical_note`、`output_format`。`department` 可填科別文字；`threshold` 表單可填數值，例如 `0.5`，送出時轉為 JSON number。這兩個保留欄位目前不參與編碼、投票或篩選。`icd_codes` 表單接受 JSON array；例如 `[{"code":"R07.9","system":"http://hl7.org/fhir/sid/icd-10-cm"}]`，需要本院已配置對應表。

### 回應欄位

| 欄位 | 內容與判讀 |
|---|---|
| `request_id` | 必須等於本次送出的 ID |
| `polished_clinical_note` | 整理後文字 |
| `snomed_codings` | 編碼 array；每項有 `concept_id`、`term`、`confidence`、`source` |
| `concept_id`／`term` | SNOMED CT 概念碼與名稱 |
| `confidence` | 投票支持度；不能當成臨床準確率 |
| `source` | 原始文字編碼為 `["TXT_NER"]`；加 ICD 證據時可能包含其來源 |
| `processing_metadata` | 流程與術語版本、投票輪數、時間；顯示時遮蔽內部模型識別值 |
| `allowed_tui` | 有指定 TUI 時，回傳採用的篩選值 |

預設病例會檢查已知胸痛概念 `29857009`、否認咳嗽未產生陽性編碼、3 輪 NER metadata 及完整欄位。修改病例、TUI 或 ICD 證據後，程式檢查回應結構及 POST／GET 一致性；其他病例的編碼正確性須另由專業人員覆核。
本院未支援 `stream` 欄位；請勿加入請求 body。

In [ ]:
#@title POST／編碼參數與執行
raw_clinical_note = "患者胸痛持續兩週，否認咳嗽。" #@param {type:"string"}
output_format = "simple" #@param ["simple", "fhir"]
language = "" #@param {type:"string"}
encounter_type = "" #@param {type:"string"}
department = "" #@param {type:"string"}
threshold = "" #@param {type:"string"}
SOAP = "" #@param {type:"string"}
tui_csv = "" #@param {type:"string"}
icd_codes_json = "" #@param {type:"string"}
name = "POST 編碼"
with test_case(name):
    require(bool(raw_clinical_note.strip()), "合成病歷不可空白")
    request_id = str(uuid4())
    coding_body = {"request_id": request_id, "raw_clinical_note": raw_clinical_note, "output_format": output_format}
    for field, value in [("language", language), ("encounter_type", encounter_type), ("SOAP", SOAP), ("department", department)]:
        if value.strip():
            coding_body[field] = value.strip()
    if tui_csv.strip():
        coding_body["tui"] = [v.strip() for v in tui_csv.split(",") if v.strip()]
    if threshold.strip():
        try:
            coding_body["threshold"] = float(threshold)
        except ValueError:
            raise RuntimeError("threshold 請填數值，例如 0.5；留白表示不送出") from None
    if icd_codes_json.strip():
        coding_body["icd_codes"] = json.loads(icd_codes_json)
        require(isinstance(coding_body["icd_codes"], list), "icd_codes 必須是 JSON array")
    require(set(coding_body) <= set(REQUEST_FIELDS), "請求包含本院不支援的欄位")
    coding_response = request("POST", "/api/v1/snomed/coding", body=coding_body)
    POST_RESULT = completed_coding_response(coding_response, request_id, coding_body, name)
    result_contract(POST_RESULT, request_id, coding_body)
    remember(coding_body, POST_RESULT)
    LAST_REQUEST_ID = request_id
    display(JSON(public_view(POST_RESULT), expanded=True))
    print("回查用 request_id：", LAST_REQUEST_ID)

## 4. GET／回查編碼結果

**路徑：`/api/v1/snomed/results/{request_id}`　｜　金鑰：需要　｜　預期：HTTP 200**

表單留白會使用上一個 POST 的 ID；也可以填入本院已完成的 ID。每次執行都會重新發送 GET。

| 欄位 | 用途 |
|---|---|
| `request_id` | 本次查詢的 ID |
| `status`／`status_code` | 已完成結果為 `completed`／`200`；其他狀態需依回傳內容處理 |
| `occurred_at` | 含時區的紀錄時間 |
| `duration_ms` | 伺服器記錄的處理時間，單位毫秒 |
| `response` | 與 POST 相同的完整編碼結果 |
| `error_detail` | 成功時為 `null` |

查詢本 notebook 送出的 ID 時，會比對 POST 與 GET 完整 JSON 一致；查詢其他 ID 時，會檢查已完成結果的欄位，無法比對先前 POST。

In [ ]:
#@title GET／回查參數與執行
lookup_request_id = "" #@param {type:"string"}
name = "GET 結果回查"
with test_case(name):
    selected_id = lookup_request_id.strip() or globals().get("LAST_REQUEST_ID")
    require(bool(selected_id), "請先完成 POST 編碼，或填入已有的 request_id")
    require(str(UUID(selected_id)) == selected_id, "request_id 必須是 UUID")
    GET_RESULT = lookup_completed(selected_id, name)

## 5. POST／FHIR 輸出

**同一編碼路徑，設定 `output_format="fhir"`。** 此區塊可單獨執行，不需先執行第 3 節。

使用新的 ID 與相同合成病例，檢查 `fhir_bundle.entry` 的 `Composition` 與 `Observation`，再以 GET 確認保存結果。`Observation.code.coding` 的 `system` 為 `http://snomed.info/sct`，`code`、`display` 必須與 SNOMED 編碼一致。

目前 `fhir_bundle` 是含 `entry` 的 API 包裝物件，沒有頂層 `resourceType="Bundle"`、`type`、病人或就醫資料；直接寫入院方 FHIR Server 前，需要依院方資源規格組裝。

In [ ]:
name = "FHIR 輸出與回查"
with test_case(name):
    fhir_id = str(uuid4())
    fhir_body = {"request_id": fhir_id, "raw_clinical_note": RAW_NOTE, "output_format": "fhir"}
    fhir_response = request("POST", "/api/v1/snomed/coding", body=fhir_body)
    checked(fhir_response, 200, name)
    FHIR_RESULT = fhir_response.json()
    result_contract(FHIR_RESULT, fhir_id, fhir_body)
    remember(fhir_body, FHIR_RESULT)
    show_json(fhir_response)
    FHIR_GET_RESULT = lookup_completed(fhir_id, name)

## 錯誤處理／逐項測試

以下故意發送無效請求。**顯示預期的 401、404、422 才是該測試通過**，不會執行有效病例編碼。

| HTTP | HIS 端處理方式 |
|---|---|
| 401 | 核對本院金鑰與 `X-API-Key`；不要自動改用其他院別 |
| 404 | 核對院別根網址及 `request_id`；未開放的路徑也會回 404 |
| 422 | 修正欄位名稱、型別或列舉值後再送出 |
| 502／503／504 | 記錄 HTTP、request_id、時間並提供給計畫維運窗口；本手冊不把這些狀態判為通過 |

實際錯誤的 `detail` 可能含服務內部資訊；此手冊僅顯示狀態及驗證欄位，不轉印內部來源。

### POST 無金鑰

省略 X-API-Key，預期 HTTP 401。

In [ ]:
name = 'POST 無金鑰'
with test_case(name):
    error_response = request('POST', "/api/v1/snomed/coding", body={"request_id": str(uuid4()), "raw_clinical_note": RAW_NOTE, "output_format": "simple"}, authenticated=False)
    checked(error_response, 401, name)
    show_error(error_response)

### GET 無金鑰

省略 X-API-Key，預期 HTTP 401。

In [ ]:
name = 'GET 無金鑰'
with test_case(name):
    error_response = request('GET', "/api/v1/snomed/results/" + str(uuid4()) + "", authenticated=False)
    checked(error_response, 401, name)
    show_error(error_response)

### 錯誤金鑰

使用明確無效的測試字串，預期 HTTP 401。

In [ ]:
name = '錯誤金鑰'
with test_case(name):
    error_response = request('GET', "/api/v1/snomed/results/" + str(uuid4()) + "", authenticated=False, headers={"X-API-Key": "invalid-test-token", "Origin": COLAB_ORIGIN})
    checked(error_response, 401, name)
    show_error(error_response)

### 查無 request_id

使用本院有效金鑰與新的隨機 UUID，預期 HTTP 404。

In [ ]:
name = '查無 request_id'
with test_case(name):
    error_response = request('GET', "/api/v1/snomed/results/" + str(uuid4()) + "")
    checked(error_response, 404, name)
    show_error(error_response)

### 不支援的欄位

加入 unknown_field，預期 HTTP 422。

In [ ]:
name = '不支援的欄位'
with test_case(name):
    error_response = request('POST', "/api/v1/snomed/coding", body={"request_id": str(uuid4()), "raw_clinical_note": RAW_NOTE, "output_format": "simple", "unknown_field": True})
    checked(error_response, 422, name)
    show_error(error_response)

### 錯誤 output_format

設定無效列舉值，預期 HTTP 422。

In [ ]:
name = '錯誤 output_format'
with test_case(name):
    error_response = request('POST', "/api/v1/snomed/coding", body={"request_id": str(uuid4()), "raw_clinical_note": RAW_NOTE, "output_format": "invalid"})
    checked(error_response, 422, name)
    show_error(error_response)

## CORS／瀏覽器預檢

`OPTIONS` 不帶金鑰，檢查是否允許 Colab Origin、對應方法與 `content-type,x-api-key`。前面的成功 POST／GET 和錯誤回應也會檢查 `Access-Control-Allow-Origin`。

Python `requests` 本身不受瀏覽器 CORS 限制。以下通過代表已檢查 **`https://colab.research.google.com`** 的 headers；院方自己的網頁 Origin 仍需另行驗證。

### POST 預檢

In [ ]:
name = "POST CORS 預檢"
with test_case(name):
    preflight("/api/v1/snomed/coding", "POST", name)

### GET 預檢

In [ ]:
name = "GET CORS 預檢"
with test_case(name):
    preflight("/api/v1/snomed/results/" + str(uuid4()), "GET", name)

## 本次測試總表

只列出**這次執行階段實際跑過**的結果。尚未執行不會算通過。可依表格找出需要重跑的區塊。這些是 API 可用性、格式及合成病例檢查，不等於臨床準確率或院方整合驗收。

In [ ]:
EXPECTED_TEST_NAMES = ['健康檢查', 'OpenAPI 公開範圍', 'POST 編碼', 'GET 結果回查', 'FHIR 輸出與回查', 'POST 無金鑰', 'GET 無金鑰', '錯誤金鑰', '查無 request_id', '不支援的欄位', '錯誤 output_format', 'POST CORS 預檢', 'GET CORS 預檢']
rows = [TESTS[name] if name in TESTS else {"項目": name, "結果": "未執行", "HTTP": "", "秒數": ""} for name in EXPECTED_TEST_NAMES]
show_table(rows)
passed = sum(row["結果"] == "通過" for row in rows)
print(f"本次 {passed}/{len(rows)} 項通過。")

## HIS 呼叫範例

正式 POST URL：`https://smartcodern.itri-nlp.tw/tph/api/v1/snomed/coding`。請在院方程式中配置本院金鑰，不要將它寫入原始碼。

```python
import os, uuid, requests
base_url = "https://smartcodern.itri-nlp.tw/tph"
headers = {"X-API-Key": os.environ["SMARTCODER_API_KEY"]}
body = {"request_id": str(uuid.uuid4()), "raw_clinical_note": "患者胸痛持續兩週，否認咳嗽。", "output_format": "simple"}
r = requests.post(base_url + "/api/v1/snomed/coding", headers=headers, json=body, timeout=240)
r.raise_for_status()
result = r.json()
saved = requests.get(base_url + "/api/v1/snomed/results/" + result["request_id"], headers=headers, timeout=60)
saved.raise_for_status()
```

此範例供院內程式配置使用；Colab 的金鑰取得方式仍是前面的隱藏輸入。若在瀏覽器前端呼叫，請先確認院方 Origin 已被允許，並避免把院級金鑰散發到公開前端。

### 使用完畢／清除金鑰

清除目前記憶體中的連線、金鑰與 API 結果。分享 notebook 前，另選「編輯 → 清除所有輸出」，或以原始公開連結提供給其他人。

In [ ]:
API_KEY = ""
COMPLETED.clear()
for variable_name in ["POST_RESULT", "GET_RESULT", "FHIR_RESULT", "FHIR_GET_RESULT", "STREAM_RESULT", "STREAM_GET_RESULT", "ICD_RESULT", "ICD_GET_RESULT", "LAST_REQUEST_ID", "coding_response", "icd_response", "fhir_response", "stream_response", "schema_response", "error_response", "event", "original", "payload"]:
    globals().pop(variable_name, None)
SESSION.close()
print("已移除金鑰變數並關閉連線。需要再測試時，請重跑初始化及金鑰設定。")